# Pun Khattgar
## Group-2 Assignment-7
### Polarization Detection, Polarization Type Classification, and Polarization Manifestation Identification

### Step 1 — Reading the CSV File

In [1]:
import pandas as pd
from pathlib import Path

file_path = Path.home() / "AI Engineering" / "PunKhattgar_PolarAnnotations_Assignment2" / "PunKhattgar_PolarAnnotations_Assignment2.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print(df.columns.tolist())

display(df.head())

Dataset shape: (600, 16)
['id', 'source', 'text', 'key_phrase', 'polarization', 'political', 'racial/ethnic', 'religious', 'gender/sexual', 'other', 'stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation']


,id,source,text,key_phrase,polarization,political,racial/ethnic,religious,gender/sexual,other,stereotype,vilification,dehumanization,extreme_language,lack_of_empathy,invalidation
0,mya_punkhattgar_1,https://www.youtube.com/watch?v=4uiLpnSZsL0 ||...,ခေတ်ပျက်ထဲမှာ တန်ဖိုးရှိတဲ့အနုပညာသည်တွေပေါ်ထွက...,NaN,0,0,0,0,0,0,0,0,0,0,0,0
1,mya_punkhattgar_2,https://www.youtube.com/watch?v=sg6wSb_DuIE ||...,မူးယစ်ဆေးဝါးမြစ်ဖျားခံရာတိုင်းရင်းသားလက်နက်ကို...,NaN,1,1,1,0,0,0,0,1,0,0,0,0
2,mya_punkhattgar_3,https://www.youtube.com/watch?v=sg6wSb_DuIE ||...,တကယ်တော့ဘိန်းမဲကိုအဆီချပြီးမြင်းခွာရွက်ခြောက်န...,NaN,0,0,0,0,0,0,0,0,0,0,0,0
3,mya_punkhattgar_4,https://www.youtube.com/watch?v=sg6wSb_DuIE ||...,ကျုပ်တို့လိုသာမန်ပြည်သူတွေနဲ့ဘာမှမသက်ဆိုင်ပါဘူ...,NaN,0,0,0,0,0,0,0,0,0,0,0,0
4,mya_punkhattgar_5,https://www.youtube.com/watch?v=sg6wSb_DuIE ||...,မှန်တယ်ဗျာ၊မူးယစ်ဆေးထုတ်လုပ်သူတွေ၊ကူးသန်းရောင်...,လူသတ်လုပ်ငန်း ||| အကုသိုလ်ကောင်,0,0,0,0,0,1,1,1,0,0,0,0


### Step 2 — Checking Data Quality

In [2]:
print("Duplicate IDs:", df["id"].duplicated().sum())
print("Missing text:", df["text"].isna().sum())
print("Missing labels:")
print(df[["polarization"]].isna().sum())

print("\nPolarization distribution:")
print(df["polarization"].value_counts(dropna=False))

print("\nDuplicate texts:", df["text"].duplicated().sum())

Duplicate IDs: 0
Missing text: 0
Missing labels:
polarization    0
dtype: int64

Polarization distribution:
polarization
0    538
1     62
Name: count, dtype: int64

Duplicate texts: 1


### Step 3 — Preparing Metadata

In [5]:
df["source_url"] = (
    df["source"].astype("string")
    .str.split(r"\s*\|\|\|", regex=True)
    .str[0]
    .str.strip()
)

df["source_type"] = (
    df["source"].astype("string")
    .str.split(r"\s*\|\|\|", regex=True)
    .str[-1]
    .str.strip()
)

df["annotator"] = "needs_verification"

df.to_csv("myPolar_full_metadata.csv", index=False)

### Step 4 — Extracting Task-Specific Datasets

In [4]:
type_cols = [
    "political",
    "racial/ethnic",
    "religious",
    "gender/sexual",
    "other"
]

manifestation_cols = [
    "stereotype",
    "vilification",
    "dehumanization",
    "extreme_language",
    "lack_of_empathy",
    "invalidation"
]

task1 = df[["id", "text", "polarization"]].copy()

task2 = df[["id", "text"] + type_cols].copy()

task3 = df[["id", "text"] + manifestation_cols].copy()

task1.to_csv("task1_detection.csv", index=False)
task2.to_csv("task2_type.csv", index=False)
task3.to_csv("task3_manifestation.csv", index=False)

### Step 5 — Train, Validation, and Test Splitting

In [8]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    task1,
    test_size=0.30,
    random_state=42,
    stratify=task1["polarization"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["polarization"]
)

print(len(train_df), len(val_df), len(test_df))
print(test_df["polarization"].value_counts())

420 90 90
polarization
0    81
1     9
Name: count, dtype: int64


## Step 6 — Task-1: Polarization Detection Model
### 6.1 Train, Validation, Test

In [9]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTest label distribution:")
print(test_df["polarization"].value_counts())

Train: 420
Validation: 90
Test: 90

Test label distribution:
polarization
0    81
1     9
Name: count, dtype: int64


### 6.2 TF-IDF + Logistic Regression

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# Create baseline model
model_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        max_features=5000
    )),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Train the model
model_lr.fit(
    train_df["text"].fillna("").astype(str),
    train_df["polarization"].astype(int)
)

print("Logistic Regression training completed!")

Logistic Regression training completed!


### 6.3 Validation Set

In [11]:
X_val = val_df["text"].fillna("").astype(str)
y_val = val_df["polarization"].astype(int)

val_pred = model_lr.predict(X_val)

print("Validation Accuracy:",
      accuracy_score(y_val, val_pred))

print("Validation F1:",
      f1_score(y_val, val_pred, zero_division=0))

print("\nValidation Classification Report:")
print(classification_report(
    y_val,
    val_pred,
    zero_division=0
))

Validation Accuracy: 0.8444444444444444
Validation F1: 0.3

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91        80
           1       0.30      0.30      0.30        10

    accuracy                           0.84        90
   macro avg       0.61      0.61      0.61        90
weighted avg       0.84      0.84      0.84        90



### 6.4 Test Set 

In [12]:
X_test = test_df["text"].fillna("").astype(str)
y_test = test_df["polarization"].astype(int)

test_pred = model_lr.predict(X_test)

print("Test Accuracy:",
      accuracy_score(y_test, test_pred))

print("\nTest Classification Report:")
print(classification_report(
    y_test,
    test_pred,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

Test Accuracy: 0.8333333333333334

Test Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.90      0.91        81
           1       0.20      0.22      0.21         9

    accuracy                           0.83        90
   macro avg       0.56      0.56      0.56        90
weighted avg       0.84      0.83      0.84        90

Confusion Matrix:
[[73  8]
 [ 7  2]]


### Step 7 — Linear SVM 

In [13]:
from sklearn.svm import LinearSVC

model_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        max_features=5000
    )),
    ("classifier", LinearSVC(
        class_weight="balanced",
        random_state=42
    ))
])

model_svm.fit(
    train_df["text"].fillna("").astype(str),
    train_df["polarization"].astype(int)
)

svm_val_pred = model_svm.predict(
    val_df["text"].fillna("").astype(str)
)

print(classification_report(
    val_df["polarization"].astype(int),
    svm_val_pred,
    zero_division=0
))

              precision    recall  f1-score   support

           0       0.90      0.91      0.91        80
           1       0.22      0.20      0.21        10

    accuracy                           0.83        90
   macro avg       0.56      0.56      0.56        90
weighted avg       0.83      0.83      0.83        90



In [12]:
results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Validation F1": f1_score(
            val_df["polarization"].astype(int),
            model_lr.predict(
                val_df["text"].fillna("").astype(str)
            ),
            zero_division=0
        )
    },
    {
        "Model": "Linear SVM",
        "Validation F1": f1_score(
            val_df["polarization"].astype(int),
            model_svm.predict(
                val_df["text"].fillna("").astype(str)
            ),
            zero_division=0
        )
    }
])

display(results.sort_values(
    "Validation F1",
    ascending=False
))

,Model,Validation F1
0,Logistic Regression,0.300000
1,Linear SVM,0.210526


## Task-2: Polarization Type Classification

In [14]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    hamming_loss
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [15]:
type_cols = [
    "political",
    "racial/ethnic",
    "religious",
    "gender/sexual",
    "other"
]

required_cols = ["id", "text", "polarization"] + type_cols

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing columns in df: {missing_cols}"
    )

print("Duplicate IDs:", df["id"].duplicated().sum())

def attach_type_labels(split_df):
    labels = df[["id"] + type_cols].copy()

    result = split_df[["id", "text"]].merge(
        labels,
        on="id",
        how="left",
        validate="one_to_one"
    )

    return result


train2 = attach_type_labels(train_df)
val2 = attach_type_labels(val_df)
test2 = attach_type_labels(test_df)

print("Train:", train2.shape)
print("Validation:", val2.shape)
print("Test:", test2.shape)

Duplicate IDs: 0
Train: (420, 7)
Validation: (90, 7)
Test: (90, 7)


In [16]:
print("Task-2 Label Distribution\n")

print(
    train2[type_cols]
    .apply(pd.to_numeric, errors="coerce")
    .sum()
    .sort_values(ascending=False)
)

print("\nTrain Label Counts:")
display(train2[type_cols].apply(pd.Series.value_counts))

Task-2 Label Distribution

political        37
other            24
racial/ethnic    18
gender/sexual    15
religious         6
dtype: int64

Train Label Counts:


,political,racial/ethnic,religious,gender/sexual,other
0,383,402,414,405,396
1,37,18,6,15,24


In [17]:
def attach_type_labels(split_df):
    labels = df[
        ["id", "polarization"] + type_cols
    ].copy()

    result = split_df[["id", "text"]].merge(
        labels,
        on="id",
        how="left",
        validate="one_to_one"
    )

    return result


train2 = attach_type_labels(train_df)
val2 = attach_type_labels(val_df)
test2 = attach_type_labels(test_df)

# Polarization ရှိသော စာကြောင်းများကိုသာ ရွေးပါ
train2_pos = train2[train2["polarization"] == 1].copy()
val2_pos = val2[val2["polarization"] == 1].copy()
test2_pos = test2[test2["polarization"] == 1].copy()

print("Polarized training rows:", len(train2_pos))
print("Polarized validation rows:", len(val2_pos))
print("Polarized test rows:", len(test2_pos))

Polarized training rows: 43
Polarized validation rows: 10
Polarized test rows: 9


In [18]:
model_task2 = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            max_features=5000
        )
    ),
    (
        "classifier",
        OneVsRestClassifier(
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    )
])

X_train2 = (
    train2_pos["text"]
    .fillna("")
    .astype(str)
)

Y_train2 = train2_pos[type_cols].astype(int)

print("Training label counts:")
print(Y_train2.sum())

print("\nTraining model...")

model_task2.fit(X_train2, Y_train2)

print("Task-2 model training completed!")

Training label counts:
political        24
racial/ethnic    14
religious         3
gender/sexual    10
other            14
dtype: int64

Training model...
Task-2 model training completed!


In [18]:
X_val2 = (
    val2_pos["text"]
    .fillna("")
    .astype(str)
)

Y_val2 = val2_pos[type_cols].astype(int)

val_pred2 = model_task2.predict(X_val2)

print("Validation Macro-F1:",
      f1_score(
          Y_val2,
          val_pred2,
          average="macro",
          zero_division=0
      ))

print("\nValidation Micro-F1:",
      f1_score(
          Y_val2,
          val_pred2,
          average="micro",
          zero_division=0
      ))

print("\nValidation Classification Report:")

print(classification_report(
    Y_val2,
    val_pred2,
    target_names=type_cols,
    zero_division=0
))

Validation Macro-F1: 0.29333333333333333

Validation Micro-F1: 0.5

Validation Classification Report:
               precision    recall  f1-score   support

    political       0.67      1.00      0.80         4
racial/ethnic       0.50      1.00      0.67         2
    religious       0.00      0.00      0.00         0
gender/sexual       0.00      0.00      0.00         3
        other       0.00      0.00      0.00         4

    micro avg       0.55      0.46      0.50        13
    macro avg       0.23      0.40      0.29        13
 weighted avg       0.28      0.46      0.35        13
  samples avg       0.48      0.45      0.43        13



In [19]:
X_test2 = (
    test2_pos["text"]
    .fillna("")
    .astype(str)
)

Y_test2 = test2_pos[type_cols].astype(int)

test_pred2 = model_task2.predict(X_test2)

macro_f1 = f1_score(
    Y_test2,
    test_pred2,
    average="macro",
    zero_division=0
)

micro_f1 = f1_score(
    Y_test2,
    test_pred2,
    average="micro",
    zero_division=0
)

exact_match = accuracy_score(
    Y_test2,
    test_pred2
)

print("Task-2 Test Results")
print("-------------------")
print("Macro-F1:", round(macro_f1, 4))
print("Micro-F1:", round(micro_f1, 4))
print("Exact-match accuracy:", round(exact_match, 4))

print("\nPer-label Results:")
print(classification_report(
    Y_test2,
    test_pred2,
    target_names=type_cols,
    zero_division=0
))

Task-2 Test Results
-------------------
Macro-F1: 0.3205
Micro-F1: 0.3333
Exact-match accuracy: 0.1111

Per-label Results:
               precision    recall  f1-score   support

    political       0.25      0.33      0.29         3
racial/ethnic       0.25      0.25      0.25         4
    religious       0.00      0.00      0.00         1
gender/sexual       1.00      0.50      0.67         2
        other       0.33      0.50      0.40         2

    micro avg       0.33      0.33      0.33        12
    macro avg       0.37      0.32      0.32        12
 weighted avg       0.37      0.33      0.33        12
  samples avg       0.22      0.28      0.24        12



In [20]:
comparison = pd.DataFrame(
    test_pred2,
    columns=[f"pred_{c}" for c in type_cols],
    index=test2_pos.index
)

comparison = comparison.join(
    test2_pos[["id", "text"] + type_cols]
)

display(comparison.head(10))

,pred_political,pred_racial/ethnic,pred_religious,pred_gender/sexual,pred_other,id,text,political,racial/ethnic,religious,gender/sexual,other
6,1,1,0,0,0,mya_punkhattgar_281,ဟာသတွေပါဗျာ ဒီလို မျိုးနဲ့ဘဲ မြန်မာပြည်သာတွေရှ...,1,0,0,0,1
10,1,0,0,0,1,mya_punkhattgar_196,မြန်မာမိန်းကလေးတွေကို အဲ့ဒီနိုင်ငံတွေကို လူကုန...,0,1,1,0,0
11,0,1,0,0,0,mya_punkhattgar_223,တကယ်တော့ Fight ချင်ရင် ယူကရိန်ကို တရာဝင်ကူညီပ...,1,0,0,0,0
18,1,0,0,0,0,mya_punkhattgar_180,ထိုင်းနိုင်ငံအစိုးရဝန်ကြီးတွေဘာကိုဆိုလိုတယ်ဆို...,0,1,0,0,0
28,0,0,0,0,0,mya_punkhattgar_195,"""ရှေ့တန်းကနေ ပြန်လာပြီ ငတ်ပြတ်နေတိုင်း တွေ့ကရာ...",0,0,0,1,0
48,1,0,0,0,1,mya_punkhattgar_276,ထိုင်းမှာလာပြီး ကိုယ့်လူမျိုး နာမည်ဖျက်နေတဲ့ ဆ...,0,1,0,0,0
50,0,0,0,0,0,mya_punkhattgar_183,ဖာနိုင်ငံ ဖာတွေးတွေးတာမဆန်းပါဘူး။,1,0,0,0,0
57,0,1,0,0,1,mya_punkhattgar_248,အော်နီပိန်းပီသပါပေတယ်၊ခုထိဖွတ်ထွက်လို့တောင်ပို...,0,0,0,0,1
81,0,1,0,1,0,mya_punkhattgar_159,တောသားတွေက မသိဘူးအဲ့လိုတွေ တကယ်တမ်းမြန်မာပြည်မ...,0,1,0,1,0


In [21]:
for i, label in enumerate(type_cols):
    actual = Y_test2[label].to_numpy()
    predicted = test_pred2[:, i]

    errors = (actual != predicted).sum()

    print(f"{label}: {errors} incorrect predictions")

political: 5 incorrect predictions
racial/ethnic: 6 incorrect predictions
religious: 1 incorrect predictions
gender/sexual: 1 incorrect predictions
other: 3 incorrect predictions


In [23]:
predictions_df = pd.DataFrame(
    test_pred2,
    columns=type_cols
)

predictions_df.insert(
    0,
    "text",
    test2_pos["text"].to_numpy()
)

predictions_df.insert(
    0,
    "id",
    test2_pos["id"].to_numpy()
)

predictions_df.to_csv(
    "task2_test_predictions.csv",
    index=False
)

print("Saved: task2_test_predictions.csv")

Saved: task2_test_predictions.csv


### Task-2: Polarization Type Classification

For Task-2, we implemented a multi-label text classification model to identify five types of polarization: political, racial/ethnic, religious, gender/sexual, and other. We used TF-IDF to transform text into numerical features and One-vs-Rest Logistic Regression as the baseline classifier. The model was trained on the polarized instances in the training split and evaluated using Macro-F1, Micro-F1, and per-label precision, recall, and F1-scores.
The dataset contains an imbalanced distribution of labels, particularly for the religious category. Therefore, the evaluation results should be interpreted cautiously because the dataset is small and some labels have very few positive examples. Further experiments with additional data and alternative classification models are needed to assess generalization.

## Task-3: Polarization Strategy Classification

In [25]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    hamming_loss
)

print("Task-3 libraries imported successfully!")


Task-3 libraries imported successfully!


In [26]:
strategy_cols = [
    "stereotype",
    "vilification",
    "dehumanization",
    "extreme_language",
    "lack_of_empathy",
    "invalidation"
]

print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

missing_cols = [
    col for col in ["id", "text", "polarization"] + strategy_cols
    if col not in df.columns
]

if missing_cols:
    print("\nMissing columns:", missing_cols)
else:
    print("\nAll Task-3 columns are available!")

print("\nDuplicate IDs:", df["id"].duplicated().sum())


Dataset shape: (600, 19)

Available columns:
['id', 'source', 'text', 'key_phrase', 'polarization', 'political', 'racial/ethnic', 'religious', 'gender/sexual', 'other', 'stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation', 'source_url', 'source_type', 'annotator']

All Task-3 columns are available!

Duplicate IDs: 0


In [27]:
def attach_strategy_labels(split_df):
    labels = df[
        ["id", "polarization"] + strategy_cols
    ].copy()

    result = split_df[["id", "text"]].merge(
        labels,
        on="id",
        how="left",
        validate="one_to_one"
    )

    return result


train3 = attach_strategy_labels(train_df)
val3 = attach_strategy_labels(val_df)
test3 = attach_strategy_labels(test_df)

print("Train rows:", len(train3))
print("Validation rows:", len(val3))
print("Test rows:", len(test3))

display(train3.head())


Train rows: 420
Validation rows: 90
Test rows: 90


,id,text,polarization,stereotype,vilification,dehumanization,extreme_language,lack_of_empathy,invalidation
0,mya_punkhattgar_587,ဥပမာအားဖြင့် မီးခတ်ကျောက်နှင့် chert ကျောက်မျာ...,0,0,0,0,0,0,0
1,mya_punkhattgar_47,အကိုရေ ကံကတကယ်ကောင်းတာဗျ🎉🎉,0,0,0,0,0,0,0
2,mya_punkhattgar_42,နောင် လာ‌ နောင်သားတွေသနားလိုက်တာ,0,0,0,0,0,0,0
3,mya_punkhattgar_561,ထိုနေ့သည် ၁၉၃၈ ခုနှစ်တွင် စတင်ပေါ်ပေါက်ခဲ့၍ ဖြ...,0,0,0,0,0,0,0
4,mya_punkhattgar_444,မြန်မာ့နတ်သမိုင်း၏ အထင်ရှားဆုံး အချိုးအကွေ့မှာ...,0,0,0,0,0,0,0


In [29]:
print("Task-3 Label Distribution")
print("-" * 35)

display(
    df[strategy_cols]
    .apply(pd.to_numeric, errors="coerce")
    .sum()
    .sort_values(ascending=False)
)

print("\nLabel value counts:")
for col in strategy_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


Task-3 Label Distribution
-----------------------------------


vilification        57
stereotype          49
extreme_language    27
invalidation        21
dehumanization      17
lack_of_empathy     17
dtype: int64


Label value counts:

stereotype
stereotype
0    551
1     49
Name: count, dtype: int64

vilification
vilification
0    543
1     57
Name: count, dtype: int64

dehumanization
dehumanization
0    583
1     17
Name: count, dtype: int64

extreme_language
extreme_language
0    573
1     27
Name: count, dtype: int64

lack_of_empathy
lack_of_empathy
0    583
1     17
Name: count, dtype: int64

invalidation
invalidation
0    579
1     21
Name: count, dtype: int64


In [30]:
train3_pos = train3[
    train3["polarization"] == 1
].copy()

val3_pos = val3[
    val3["polarization"] == 1
].copy()

test3_pos = test3[
    test3["polarization"] == 1
].copy()

print("Polarized training rows:", len(train3_pos))
print("Polarized validation rows:", len(val3_pos))
print("Polarized test rows:", len(test3_pos))

print("\nTraining label counts:")
display(train3_pos[strategy_cols].sum())


Polarized training rows: 43
Polarized validation rows: 10
Polarized test rows: 9

Training label counts:


stereotype          25
vilification        30
dehumanization       6
extreme_language    11
lack_of_empathy      8
invalidation        10
dtype: int64

In [31]:

X_train3 = (
    train3_pos["text"]
    .fillna("")
    .astype(str)
)

Y_train3 = (
    train3_pos[strategy_cols]
    .apply(pd.to_numeric, errors="coerce")
)

if Y_train3.isna().any().any():
    raise ValueError("Task-3 labels contain missing or invalid values.")

if not Y_train3.isin([0, 1]).all().all():
    raise ValueError("Task-3 labels must contain only 0 and 1.")

if Y_train3.nunique().eq(1).any():
    print("Warning: Some labels contain only one class:")
    print(Y_train3.nunique()[Y_train3.nunique().eq(1)])

    raise ValueError(
        "At least one label has only one class in training. "
        "Check the label distribution before fitting."
    )

model_task3 = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            max_features=5000
        )
    ),
    (
        "classifier",
        OneVsRestClassifier(
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    )
])

print("Training Task-3 model...")

model_task3.fit(
    X_train3,
    Y_train3.astype(int)
)

print("Task-3 model trained successfully!")


Training Task-3 model...
Task-3 model trained successfully!


In [32]:
X_val3 = (
    val3_pos["text"]
    .fillna("")
    .astype(str)
)

Y_val3 = val3_pos[strategy_cols].astype(int)

val_pred3 = model_task3.predict(X_val3)

print(
    "Validation Macro-F1:",
    round(
        f1_score(
            Y_val3,
            val_pred3,
            average="macro",
            zero_division=0
        ),
        4
    )
)

print(
    "\nValidation Micro-F1:",
    round(
        f1_score(
            Y_val3,
            val_pred3,
            average="micro",
            zero_division=0
        ),
        4
    )
)

print("\nValidation Classification Report:")

print(
    classification_report(
        Y_val3,
        val_pred3,
        target_names=strategy_cols,
        zero_division=0
    )
)


Validation Macro-F1: 0.1439

Validation Micro-F1: 0.3125

Validation Classification Report:
                  precision    recall  f1-score   support

      stereotype       0.33      0.40      0.36         5
    vilification       0.38      0.75      0.50         4
  dehumanization       0.00      0.00      0.00         1
extreme_language       0.00      0.00      0.00         4
 lack_of_empathy       0.00      0.00      0.00         3
    invalidation       0.00      0.00      0.00         0

       micro avg       0.33      0.29      0.31        17
       macro avg       0.12      0.19      0.14        17
    weighted avg       0.19      0.29      0.22        17
     samples avg       0.30      0.24      0.24        17



In [33]:
X_test3 = (
    test3_pos["text"]
    .fillna("")
    .astype(str)
)

Y_test3 = test3_pos[strategy_cols].astype(int)

test_pred3 = model_task3.predict(X_test3)

macro_f1_3 = f1_score(
    Y_test3,
    test_pred3,
    average="macro",
    zero_division=0
)

micro_f1_3 = f1_score(
    Y_test3,
    test_pred3,
    average="micro",
    zero_division=0
)

exact_match_3 = accuracy_score(
    Y_test3,
    test_pred3
)

hamming_3 = hamming_loss(
    Y_test3,
    test_pred3
)

print("Task-3 Test Results")
print("-" * 35)
print("Macro-F1:", round(macro_f1_3, 4))
print("Micro-F1:", round(micro_f1_3, 4))
print("Exact-match accuracy:", round(exact_match_3, 4))
print("Hamming loss:", round(hamming_3, 4))

print("\nPer-label Classification Report:")

print(
    classification_report(
        Y_test3,
        test_pred3,
        target_names=strategy_cols,
        zero_division=0
    )
)


Task-3 Test Results
-----------------------------------
Macro-F1: 0.2711
Micro-F1: 0.5789
Exact-match accuracy: 0.1111
Hamming loss: 0.2963

Per-label Classification Report:
                  precision    recall  f1-score   support

      stereotype       1.00      0.62      0.77         8
    vilification       0.75      1.00      0.86         6
  dehumanization       0.00      0.00      0.00         2
extreme_language       0.00      0.00      0.00         1
 lack_of_empathy       0.00      0.00      0.00         1
    invalidation       0.00      0.00      0.00         2

       micro avg       0.61      0.55      0.58        20
       macro avg       0.29      0.27      0.27        20
    weighted avg       0.62      0.55      0.56        20
     samples avg       0.63      0.51      0.51        20



In [34]:
comparison3 = test3_pos[
    ["id", "text"] + strategy_cols
].copy()

for i, label in enumerate(strategy_cols):
    comparison3[f"pred_{label}"] = test_pred3[:, i]

display(comparison3.head(10))


,id,text,stereotype,vilification,dehumanization,extreme_language,lack_of_empathy,invalidation,pred_stereotype,pred_vilification,pred_dehumanization,pred_extreme_language,pred_lack_of_empathy,pred_invalidation
6,mya_punkhattgar_281,ဟာသတွေပါဗျာ ဒီလို မျိုးနဲ့ဘဲ မြန်မာပြည်သာတွေရှ...,1,1,0,0,0,1,1,1,0,0,0,0
10,mya_punkhattgar_196,မြန်မာမိန်းကလေးတွေကို အဲ့ဒီနိုင်ငံတွေကို လူကုန...,1,1,0,0,1,0,1,1,0,0,0,0
11,mya_punkhattgar_223,တကယ်တော့ Fight ချင်ရင် ယူကရိန်ကို တရာဝင်ကူညီပ...,0,0,0,0,0,0,0,0,0,1,0,0
18,mya_punkhattgar_180,ထိုင်းနိုင်ငံအစိုးရဝန်ကြီးတွေဘာကိုဆိုလိုတယ်ဆို...,1,1,0,0,0,0,1,1,0,0,0,0
28,mya_punkhattgar_195,"""ရှေ့တန်းကနေ ပြန်လာပြီ ငတ်ပြတ်နေတိုင်း တွေ့ကရာ...",1,0,0,0,0,0,0,1,0,1,0,0
48,mya_punkhattgar_276,ထိုင်းမှာလာပြီး ကိုယ့်လူမျိုး နာမည်ဖျက်နေတဲ့ ဆ...,1,1,1,0,0,0,0,1,0,1,0,1
50,mya_punkhattgar_183,ဖာနိုင်ငံ ဖာတွေးတွေးတာမဆန်းပါဘူး။,1,1,0,1,0,0,1,1,0,0,0,0
57,mya_punkhattgar_248,အော်နီပိန်းပီသပါပေတယ်၊ခုထိဖွတ်ထွက်လို့တောင်ပို...,1,1,1,0,0,1,0,1,0,0,0,0
81,mya_punkhattgar_159,တောသားတွေက မသိဘူးအဲ့လိုတွေ တကယ်တမ်းမြန်မာပြည်မ...,1,0,0,0,0,0,1,1,0,0,1,0


In [35]:
predictions3 = pd.DataFrame(
    test_pred3,
    columns=strategy_cols
)

predictions3.insert(
    0,
    "text",
    test3_pos["text"].to_numpy()
)

predictions3.insert(
    0,
    "id",
    test3_pos["id"].to_numpy()
)

predictions3.to_csv(
    "task3_test_predictions.csv",
    index=False
)

print("Saved: task3_test_predictions.csv")


Saved: task3_test_predictions.csv


## Final Evaluation, Model Comparison and Report Preparation

In [36]:
important_names = [
    "train_df", "val_df", "test_df",
    "model_task1", "model_task2", "model_task3",
    "y_test", "y_pred",
    "y_test_task1", "y_pred_task1",
    "y_test_task2", "y_pred_task2",
    "y_test_task3", "y_pred_task3"
]

for name in important_names:
    print(f"{name}: {name in globals()}")


train_df: True
val_df: True
test_df: True
model_task1: False
model_task2: True
model_task3: True
y_test: True
y_pred: False
y_test_task1: False
y_pred_task1: False
y_test_task2: False
y_pred_task2: False
y_test_task3: False
y_pred_task3: False


In [37]:
print("=== Dataset Columns ===")
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

print("\n=== Task-2 Model ===")
print(model_task2)

print("\n=== Task-3 Model ===")
print(model_task3)


=== Dataset Columns ===
Train columns: ['id', 'text', 'polarization']
Test columns: ['id', 'text', 'polarization']

=== Task-2 Model ===
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                ('classifier',
                 OneVsRestClassifier(estimator=LogisticRegression(class_weight='balanced',
                                                                  max_iter=1000,
                                                                  random_state=42)))])

=== Task-3 Model ===
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                ('classifier',
                 OneVsRestClassifier(estimator=LogisticRegression(class_weight='balanced',
                                                                  max_iter=1000,
                                                                  random_state=42)))])


In [38]:
for name, obj in list(globals().items()):
    if any(word in name.lower() for word in
           ["model", "classifier", "pipeline", "prediction", "pred"]):
        if not name.startswith("_"):
            print(name, "->", type(obj).__name__)


Pipeline -> ABCMeta
model_lr -> Pipeline
val_pred -> ndarray
test_pred -> ndarray
model_svm -> Pipeline
svm_val_pred -> ndarray
OneVsRestClassifier -> type
model_task2 -> Pipeline
val_pred2 -> ndarray
test_pred2 -> ndarray
predicted -> ndarray
predictions_df -> DataFrame
model_task3 -> Pipeline
val_pred3 -> ndarray
test_pred3 -> ndarray
predictions3 -> DataFrame


In [39]:
import numpy as np

variables_to_check = [
    "test_pred",
    "test_pred2",
    "test_pred3",
    "predicted",
]

for name in variables_to_check:
    if name in globals():
        arr = np.asarray(globals()[name])
        print(f"\n{name}")
        print("Shape:", arr.shape)
        print("Data type:", arr.dtype)
        print("First 3 predictions:")
        print(arr[:3])

print("\n=== Task-2 Prediction DataFrame ===")
if "predictions_df" in globals():
    print("Columns:", predictions_df.columns.tolist())
    display(predictions_df.head(3))

print("\n=== Task-3 Prediction DataFrame ===")
if "predictions3" in globals():
    print("Columns:", predictions3.columns.tolist())
    display(predictions3.head(3))



test_pred
Shape: (90,)
Data type: int64
First 3 predictions:
[0 0 0]

test_pred2
Shape: (9, 5)
Data type: int64
First 3 predictions:
[[1 1 0 0 0]
 [1 0 0 0 1]
 [0 1 0 0 0]]

test_pred3
Shape: (9, 6)
Data type: int64
First 3 predictions:
[[1 1 0 0 0 0]
 [1 1 0 0 0 0]
 [0 0 0 1 0 0]]

predicted
Shape: (9,)
Data type: int64
First 3 predictions:
[0 1 0]

=== Task-2 Prediction DataFrame ===
Columns: ['id', 'text', 'political', 'racial/ethnic', 'religious', 'gender/sexual', 'other']


,id,text,political,racial/ethnic,religious,gender/sexual,other
0,mya_punkhattgar_281,ဟာသတွေပါဗျာ ဒီလို မျိုးနဲ့ဘဲ မြန်မာပြည်သာတွေရှ...,1,1,0,0,0
1,mya_punkhattgar_196,မြန်မာမိန်းကလေးတွေကို အဲ့ဒီနိုင်ငံတွေကို လူကုန...,1,0,0,0,1
2,mya_punkhattgar_223,တကယ်တော့ Fight ချင်ရင် ယူကရိန်ကို တရာဝင်ကူညီပ...,0,1,0,0,0



=== Task-3 Prediction DataFrame ===
Columns: ['id', 'text', 'stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation']


,id,text,stereotype,vilification,dehumanization,extreme_language,lack_of_empathy,invalidation
0,mya_punkhattgar_281,ဟာသတွေပါဗျာ ဒီလို မျိုးနဲ့ဘဲ မြန်မာပြည်သာတွေရှ...,1,1,0,0,0,0
1,mya_punkhattgar_196,မြန်မာမိန်းကလေးတွေကို အဲ့ဒီနိုင်ငံတွေကို လူကုန...,1,1,0,0,0,0
2,mya_punkhattgar_223,တကယ်တော့ Fight ချင်ရင် ယူကရိန်ကို တရာဝင်ကူညီပ...,0,0,0,1,0,0


In [40]:
# 1. Check the sizes of all dataset splits
print("=== Dataset Sizes ===")
for name in ["train_df", "val_df", "test_df"]:
    if name in globals():
        print(f"{name}: {len(globals()[name])} rows")

# 2. Check prediction sizes
print("\n=== Prediction Shapes ===")
for name in ["test_pred", "test_pred2", "test_pred3"]:
    if name in globals():
        print(f"{name}: {np.asarray(globals()[name]).shape}")

# 3. Find possible test-label variables
print("\n=== Possible Test Label Variables ===")
for name, obj in list(globals().items()):
    if any(term in name.lower() for term in
           ["y_test", "ytest", "test_labels", "test_targets"]):
        if not name.startswith("_"):
            shape = getattr(obj, "shape", None)
            print(f"{name}: {shape}")

# 4. Check available DataFrames that may contain labels
print("\n=== Available DataFrames ===")
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        if any(term in name.lower() for term in
               ["task", "test", "label", "df"]):
            print(f"{name}: rows={len(obj)}, columns={obj.columns.tolist()}")


=== Dataset Sizes ===
train_df: 420 rows
val_df: 90 rows
test_df: 90 rows

=== Prediction Shapes ===
test_pred: (90,)
test_pred2: (9, 5)
test_pred3: (9, 6)

=== Possible Test Label Variables ===
y_test: (90,)
Y_test2: (9, 5)
Y_test3: (9, 6)

=== Available DataFrames ===
df: rows=600, columns=['id', 'source', 'text', 'key_phrase', 'polarization', 'political', 'racial/ethnic', 'religious', 'gender/sexual', 'other', 'stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation', 'source_url', 'source_type', 'annotator']
task1: rows=600, columns=['id', 'text', 'polarization']
task2: rows=600, columns=['id', 'text', 'political', 'racial/ethnic', 'religious', 'gender/sexual', 'other']
task3: rows=600, columns=['id', 'text', 'stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation']
train_df: rows=420, columns=['id', 'text', 'polarization']
temp_df: rows=180, columns=['id', 'text', 'polarization']
val_df: rows=

In [41]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ==========================================
# TASK 1: Binary Classification
# ==========================================
print("=" * 60)
print("TASK 1 — BINARY POLARIZATION CLASSIFICATION")
print("=" * 60)

print("Test labels:", len(y_test))
print("Predictions:", len(test_pred))

if len(y_test) == len(test_pred):
    print("Accuracy :", round(accuracy_score(y_test, test_pred), 4))
    print("Precision:", round(
        precision_score(y_test, test_pred, average="binary", zero_division=0), 4
    ))
    print("Recall   :", round(
        recall_score(y_test, test_pred, average="binary", zero_division=0), 4
    ))
    print("F1-score :", round(
        f1_score(y_test, test_pred, average="binary", zero_division=0), 4
    ))
    print("\nClassification Report:")
    print(classification_report(y_test, test_pred, zero_division=0))
else:
    print("WARNING: Test labels and predictions have different lengths.")


# ==========================================
# TASK 2: Multi-label Polarization Types
# ==========================================
print("\n" + "=" * 60)
print("TASK 2 — POLARIZATION TYPE CLASSIFICATION")
print("=" * 60)

print("Actual labels:", Y_test2.shape)
print("Predictions  :", test_pred2.shape)

# Verify that prediction rows match the test IDs
if "predictions_df" in globals() and "test2_pos" in globals():
    print(
        "IDs aligned:",
        predictions_df["id"].reset_index(drop=True).equals(
            test2_pos["id"].reset_index(drop=True)
        )
    )

if Y_test2.shape == test_pred2.shape:
    print("Macro-F1:", round(
        f1_score(Y_test2, test_pred2, average="macro", zero_division=0), 4
    ))
    print("Micro-F1:", round(
        f1_score(Y_test2, test_pred2, average="micro", zero_division=0), 4
    ))
    print("Macro Precision:", round(
        precision_score(Y_test2, test_pred2, average="macro", zero_division=0), 4
    ))
    print("Macro Recall:", round(
        recall_score(Y_test2, test_pred2, average="macro", zero_division=0), 4
    ))
    print("\nClassification Report:")
    print(classification_report(
        Y_test2, test_pred2,
        target_names=Y_test2.columns.tolist(),
        zero_division=0
    ))
else:
    print("WARNING: Task-2 label and prediction shapes do not match.")


# ==========================================
# TASK 3: Multi-label Polarization Strategies
# ==========================================
print("\n" + "=" * 60)
print("TASK 3 — POLARIZATION STRATEGY CLASSIFICATION")
print("=" * 60)

print("Actual labels:", Y_test3.shape)
print("Predictions  :", test_pred3.shape)

if "predictions3" in globals() and "test3_pos" in globals():
    print(
        "IDs aligned:",
        predictions3["id"].reset_index(drop=True).equals(
            test3_pos["id"].reset_index(drop=True)
        )
    )

if Y_test3.shape == test_pred3.shape:
    print("Macro-F1:", round(
        f1_score(Y_test3, test_pred3, average="macro", zero_division=0), 4
    ))
    print("Micro-F1:", round(
        f1_score(Y_test3, test_pred3, average="micro", zero_division=0), 4
    ))
    print("Macro Precision:", round(
        precision_score(Y_test3, test_pred3, average="macro", zero_division=0), 4
    ))
    print("Macro Recall:", round(
        recall_score(Y_test3, test_pred3, average="macro", zero_division=0), 4
    ))
    print("\nClassification Report:")
    print(classification_report(
        Y_test3, test_pred3,
        target_names=Y_test3.columns.tolist(),
        zero_division=0
    ))
else:
    print("WARNING: Task-3 label and prediction shapes do not match.")


TASK 1 — BINARY POLARIZATION CLASSIFICATION
Test labels: 90
Predictions: 90
Accuracy : 0.8333
Precision: 0.2
Recall   : 0.2222
F1-score : 0.2105

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.90      0.91        81
           1       0.20      0.22      0.21         9

    accuracy                           0.83        90
   macro avg       0.56      0.56      0.56        90
weighted avg       0.84      0.83      0.84        90


TASK 2 — POLARIZATION TYPE CLASSIFICATION
Actual labels: (9, 5)
Predictions  : (9, 5)
IDs aligned: True
Macro-F1: 0.3205
Micro-F1: 0.3333
Macro Precision: 0.3667
Macro Recall: 0.3167

Classification Report:
               precision    recall  f1-score   support

    political       0.25      0.33      0.29         3
racial/ethnic       0.25      0.25      0.25         4
    religious       0.00      0.00      0.00         1
gender/sexual       1.00      0.50      0.67         2
        other       

# POLAR Classification: Experimental Results and Discussion

## 1. Introduction

This project explores polarization classification using a TF-IDF and Logistic Regression baseline. The experiments consist of three tasks: binary polarization classification, multi-label polarization type classification, and multi-label polarization strategy classification.

## 2. Methodology

TF-IDF was used to transform textual data into numerical feature vectors, with a maximum of 5,000 features and unigram-bigram combinations. Logistic Regression with balanced class weights was used as the baseline classifier. The multi-label tasks used a One-vs-Rest classification strategy.

The dataset was divided into training, validation, and test sets containing 420, 90, and 90 samples, respectively.

## 3. Experimental Results

### Task 1: Binary Polarization Classification

The binary classification model achieved an accuracy of 83.33%. However, its F1-score for the positive polarization class was only 0.2105. The test set contained 81 negative samples and 9 positive samples, indicating substantial class imbalance.

These results suggest that accuracy alone does not adequately represent the model's ability to identify polarized content.

### Task 2: Multi-label Polarization Type Classification

The model achieved a macro-F1 score of 0.3205 and a micro-F1 score of 0.3333 on the evaluated subset of nine samples.

Performance varied across labels. The gender/sexual category achieved an F1-score of 0.67, while the religious category achieved an F1-score of 0.00. These results suggest that the model's performance differs across polarization types.

### Task 3: Multi-label Polarization Strategy Classification

The model achieved a macro-F1 score of 0.2711 and a micro-F1 score of 0.5789 on the evaluated subset of nine samples.

The model performed comparatively well on stereotype and vilification, with F1-scores of 0.77 and 0.86, respectively. However, it did not correctly identify positive examples for several other strategy labels in this evaluation.

The difference between macro-F1 and micro-F1 indicates uneven performance across labels.

## 4. Discussion

The experiments demonstrate that TF-IDF and Logistic Regression provide a useful baseline for polarization classification. However, class imbalance and differences in label frequency affect model performance.

The results also highlight the importance of evaluating each label individually rather than relying exclusively on a single aggregate metric. Macro-F1 is useful for assessing performance across labels, while micro-F1 summarizes performance across all label decisions.

Importantly, the Task 2 and Task 3 results were calculated on only nine test samples, whereas Task 1 used 90 test samples. Therefore, the results are not directly comparable. A complete evaluation using all 90 test samples is necessary before drawing firm conclusions about the relative performance of the three tasks.

## 5. Limitations and Future Improvements

Several improvements could be explored:

1. Evaluate Task 2 and Task 3 on the complete test set.
2. Investigate class imbalance and the limited number of examples for certain labels.
3. Tune model hyperparameters using the validation set.
4. Explore alternative text representations and classification models.
5. Examine per-label precision, recall, and F1-score to identify weaknesses.

## 6. Conclusion

This project established a TF-IDF and Logistic Regression baseline for three polarization classification tasks. The initial results show that the model can identify some polarization categories and strategies more effectively than others. Nevertheless, class imbalance and the small evaluation subsets limit the reliability of the findings. Evaluating all multi-label tasks on the complete test set will provide a more reliable basis for final conclusions.